# Fuzzy Logic Braking Controller

This notebook implements the **Fuzzy Logic Braking Controller** described in the case study using Python and the **scikit-fuzzy** library.

**Inputs**
- Distance to Obstacle: 0–100 m
- Relative Speed: 0–120 km/h
- Road Friction Coefficient: 0–1

**Output**
- Braking Force: 0–100 %


### Step 1: Install Required Libraries


In [ ]:
!pip install scikit-fuzzy matplotlib numpy


### Step 2: Import Libraries


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt


### Step 3: Define Input and Output Variables


In [ ]:
# Input 1: Distance to Obstacle (m)
distance = ctrl.Antecedent(np.arange(0, 101, 1), 'distance')

# Input 2: Relative Speed (km/h)
speed = ctrl.Antecedent(np.arange(0, 121, 1), 'speed')

# Input 3: Road Friction Coefficient
friction = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'friction')

# Output: Braking Force (%)
braking = ctrl.Consequent(np.arange(0, 101, 1), 'braking')


### Step 4: Define Membership Functions

Triangular membership functions are used for all inputs and the output, as specified in the case study.


#### Distance to Obstacle


In [ ]:
distance['near'] = fuzz.trimf(distance.universe, [0, 0, 40])
distance['medium'] = fuzz.trimf(distance.universe, [20, 50, 80])
distance['far'] = fuzz.trimf(distance.universe, [60, 100, 100])


In [ ]:
distance.view()
plt.show()


#### Relative Speed


In [ ]:
speed['slow'] = fuzz.trimf(speed.universe, [0, 0, 50])
speed['medium'] = fuzz.trimf(speed.universe, [30, 60, 90])
speed['fast'] = fuzz.trimf(speed.universe, [70, 120, 120])


In [ ]:
speed.view()
plt.show()


#### Road Friction


In [ ]:
friction['low'] = fuzz.trimf(friction.universe, [0, 0, 0.4])
friction['medium'] = fuzz.trimf(friction.universe, [0.2, 0.5, 0.8])
friction['high'] = fuzz.trimf(friction.universe, [0.6, 1, 1])


In [ ]:
friction.view()
plt.show()


#### Braking Force


In [ ]:
braking['low'] = fuzz.trimf(braking.universe, [0, 0, 30])
braking['medium'] = fuzz.trimf(braking.universe, [20, 50, 80])
braking['high'] = fuzz.trimf(braking.universe, [60, 100, 100])


In [ ]:
braking.view()
plt.show()


### Step 5: Membership Function Summary

| Variable | Linguistic Terms | Range |
|---|---|---|
| Distance to Obstacle | Near, Medium, Far | 0–100 m |
| Relative Speed | Slow, Medium, Fast | 0–120 km/h |
| Road Friction | Low, Medium, High | 0–1 |
| Braking Force | Low, Medium, High | 0–100 % |


### Step 6: Create Rule Base

The Mamdani rule base uses the eight rules from the case study. For an AND condition, scikit-fuzzy uses the minimum of the antecedent memberships.


In [ ]:
# R1: Near distance and Fast speed -> High braking
rule1 = ctrl.Rule(
    distance['near'] & speed['fast'],
    braking['high']
)

# R2: Near distance and Medium speed -> Medium braking
rule2 = ctrl.Rule(
    distance['near'] & speed['medium'],
    braking['medium']
)

# R3: Medium distance and Fast speed -> High braking
rule3 = ctrl.Rule(
    distance['medium'] & speed['fast'],
    braking['high']
)

# R4: Medium distance and Medium speed -> Medium braking
rule4 = ctrl.Rule(
    distance['medium'] & speed['medium'],
    braking['medium']
)

# R5: Far distance and Slow speed -> Low braking
rule5 = ctrl.Rule(
    distance['far'] & speed['slow'],
    braking['low']
)

# R6: Far distance and Fast speed -> Medium braking
rule6 = ctrl.Rule(
    distance['far'] & speed['fast'],
    braking['medium']
)

# R7: Low road friction -> High braking
rule7 = ctrl.Rule(
    friction['low'],
    braking['high']
)

# R8: Near distance and Low road friction -> High braking
rule8 = ctrl.Rule(
    distance['near'] & friction['low'],
    braking['high']
)


### Step 7: Create Fuzzy Controller


In [ ]:
braking_ctrl = ctrl.ControlSystem([
    rule1,
    rule2,
    rule3,
    rule4,
    rule5,
    rule6,
    rule7,
    rule8
])

system = ctrl.ControlSystemSimulation(braking_ctrl)


### Step 8: Provide Sensor Values

The worked example in the case study uses:

- Distance = 20 m
- Relative Speed = 65 km/h
- Road Friction = 0.3


In [ ]:
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3


### Step 9: Run Fuzzy Inference


In [ ]:
system.compute()


In [ ]:
print("Calculated Braking Force:", round(system.output['braking'], 2), "%")


### Step 10: Display Result


In [ ]:
print("Recommended Braking Force:")
print(round(system.output['braking'], 2), "%")


### Step 11: Show Defuzzification Graph


In [ ]:
braking.view(sim=system)
plt.show()


### Step 12: Check Fuzzified Sensor Values

For the worked example, the active membership values are:

- Distance = 20 m → Near = 0.5
- Speed = 65 km/h → Medium = 0.833
- Friction = 0.3 → Low = 0.25, Medium = 0.333


In [ ]:
D = 20
V = 65
mu = 0.3

print("Distance:")
print("  Near   =", fuzz.interp_membership(distance.universe, distance['near'].mf, D))
print("  Medium =", fuzz.interp_membership(distance.universe, distance['medium'].mf, D))
print("  Far    =", fuzz.interp_membership(distance.universe, distance['far'].mf, D))

print("\nSpeed:")
print("  Slow   =", fuzz.interp_membership(speed.universe, speed['slow'].mf, V))
print("  Medium =", fuzz.interp_membership(speed.universe, speed['medium'].mf, V))
print("  Fast   =", fuzz.interp_membership(speed.universe, speed['fast'].mf, V))

print("\nFriction:")
print("  Low    =", fuzz.interp_membership(friction.universe, friction['low'].mf, mu))
print("  Medium =", fuzz.interp_membership(friction.universe, friction['medium'].mf, mu))
print("  High   =", fuzz.interp_membership(friction.universe, friction['high'].mf, mu))


### Step 13: Test Different Sensor Conditions

Different sensor values can be tested to observe how the fuzzy controller changes the braking force.


#### Test Case 1 — Near obstacle, high speed, good grip


In [ ]:
system = ctrl.ControlSystemSimulation(braking_ctrl)

system.input['distance'] = 10
system.input['speed'] = 90
system.input['friction'] = 0.8

system.compute()

print("Distance = 10 m")
print("Speed = 90 km/h")
print("Friction = 0.8")
print("Braking Force =", round(system.output['braking'], 2), "%")


#### Test Case 2 — Medium distance, medium speed


In [ ]:
system = ctrl.ControlSystemSimulation(braking_ctrl)

system.input['distance'] = 50
system.input['speed'] = 60
system.input['friction'] = 0.8

system.compute()

print("Distance = 50 m")
print("Speed = 60 km/h")
print("Friction = 0.8")
print("Braking Force =", round(system.output['braking'], 2), "%")


#### Test Case 3 — Far obstacle, slow speed


In [ ]:
system = ctrl.ControlSystemSimulation(braking_ctrl)

system.input['distance'] = 90
system.input['speed'] = 20
system.input['friction'] = 0.8

system.compute()

print("Distance = 90 m")
print("Speed = 20 km/h")
print("Friction = 0.8")
print("Braking Force =", round(system.output['braking'], 2), "%")


#### Test Case 4 — Slippery road


In [ ]:
system = ctrl.ControlSystemSimulation(braking_ctrl)

system.input['distance'] = 50
system.input['speed'] = 40
system.input['friction'] = 0.1

system.compute()

print("Distance = 50 m")
print("Speed = 40 km/h")
print("Friction = 0.1")
print("Braking Force =", round(system.output['braking'], 2), "%")


### Step 14: Apply Different Defuzzification Methods

The scikit-fuzzy controller can compare different defuzzification methods:

| Method | Meaning |
|---|---|
| Centroid | Balance point of the aggregated fuzzy output |
| Bisector | Point dividing the aggregated area into two equal areas |
| MOM | Mean of Maximum |
| SOM | Smallest of Maximum |
| LOM | Largest of Maximum |

The case study's worked calculation gives approximately **66.7%** using its weighted-average calculation.


#### 1. Centroid


In [ ]:
braking.defuzzify_method = 'centroid'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("Centroid =", round(system.output['braking'], 2), "%")
braking.view(sim=system)
plt.show()


#### 2. Bisector


In [ ]:
braking.defuzzify_method = 'bisector'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("Bisector =", round(system.output['braking'], 2), "%")
braking.view(sim=system)
plt.show()


#### 3. MOM — Mean of Maximum


In [ ]:
braking.defuzzify_method = 'mom'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("MOM =", round(system.output['braking'], 2), "%")
braking.view(sim=system)
plt.show()


#### 4. SOM — Smallest of Maximum


In [ ]:
braking.defuzzify_method = 'som'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("SOM =", round(system.output['braking'], 2), "%")
braking.view(sim=system)
plt.show()


#### 5. LOM — Largest of Maximum


In [ ]:
braking.defuzzify_method = 'lom'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("LOM =", round(system.output['braking'], 2), "%")
braking.view(sim=system)
plt.show()


### Step 15: Compare All Defuzzification Methods


In [ ]:
methods = ['centroid', 'bisector', 'mom', 'som', 'lom']
results = {}

for method in methods:
    braking.defuzzify_method = method

    system = ctrl.ControlSystemSimulation(braking_ctrl)

    system.input['distance'] = 20
    system.input['speed'] = 65
    system.input['friction'] = 0.3

    system.compute()
    results[method] = system.output['braking']

for method, value in results.items():
    print(f"{method.capitalize():10s}: {value:.2f} %")


### Step 16: Plot Defuzzification Comparison


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results.keys(), results.values())
plt.xlabel("Defuzzification Method")
plt.ylabel("Braking Force (%)")
plt.title("Braking Force for Different Defuzzification Methods")
plt.grid(axis='y', alpha=0.3)
plt.show()


### Step 17: Manual Weighted-Average Calculation from the Case Study

For the worked example, the case study identifies:

- Medium output strength = 0.5, peak = 50
- High output strength = 0.25, peak = 100

Therefore:

**Σw = 0.75**

**Σ(w × peak) = 50.0**

**y\* = 50.0 / 0.75 ≈ 66.7%**

This is the weighted-average calculation presented in the original case study. The scikit-fuzzy implementation above uses its selected defuzzification method directly.


In [ ]:
w_medium = 0.5
w_high = 0.25

peak_medium = 50
peak_high = 100

weighted_sum = (w_medium * peak_medium) + (w_high * peak_high)
sum_weights = w_medium + w_high

manual_result = weighted_sum / sum_weights

print("Manual weighted-average result =", round(manual_result, 2), "%")


### Conclusion

The fuzzy braking controller takes **distance to obstacle, relative speed, and road friction** as inputs and produces a graded **braking force**.

The notebook implements the Mamdani fuzzy inference system, membership functions, eight-rule knowledge base, sensor test cases, output visualization, and comparison of different defuzzification methods.


In [ ]:
# Final example result using the default/selected controller setup

braking.defuzzify_method = 'centroid'

system = ctrl.ControlSystemSimulation(braking_ctrl)
system.input['distance'] = 20
system.input['speed'] = 65
system.input['friction'] = 0.3
system.compute()

print("Final Braking Force (Centroid):", round(system.output['braking'], 2), "%")
